## chargement des 4 résumés par zone

On ouvre les 4 fichiers produits par le notebook d'agrégation, dans le dossier `agg_data` :
- `yellow_zone.csv` : 24 colonnes (numéro de zone + 23 totaux) ;
- `green_zone.csv` : 24 colonnes ;
- `fhv_zone.csv` : 13 colonnes (numéro de zone + 12 totaux) ;
- `hvfhv_zone.csv` : 30 colonnes (numéro de zone + 29 totaux).

Pour chacun, on affiche le nombre de zones, le nombre de colonnes et le nombre total de courses. Ce sont les chiffres de référence pour vérifier ensuite que l'intégration ne perd rien.

**Effet sur les données :** les 4 résumés sont en mémoire. Chacun ne contient que les zones où le service a au moins une course, donc ils n'ont pas tous le même nombre de lignes.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings("ignore")

AGG_DIR = os.path.join(os.getcwd(), "agg_data")
ZONES_CSV = r"C:\Users\DELL\Desktop\esprit\data_warehouse\projet\TLC\taxi_zone_lookup.csv"

yellow_zone = pd.read_csv(os.path.join(AGG_DIR, "yellow_zone.csv"))
green_zone = pd.read_csv(os.path.join(AGG_DIR, "green_zone.csv"))
fhv_zone = pd.read_csv(os.path.join(AGG_DIR, "fhv_zone.csv"))
hvfhv_zone = pd.read_csv(os.path.join(AGG_DIR, "hvfhv_zone.csv"))

for nom, df in [("yellow", yellow_zone), ("green", green_zone), ("fhv", fhv_zone), ("hvfhv", hvfhv_zone)]:
    print(nom, ":", df.shape[0], "zones,", df.shape[1], "colonnes,", df[nom + "_n"].sum(), "courses")

yellow : 259 zones, 24 colonnes, 14269897 courses
green : 240 zones, 24 colonnes, 208490 courses
fhv : 260 zones, 13 colonnes, 1044984 courses
hvfhv : 259 zones, 30 colonnes, 4048744 courses


## la table de référence des zones

On charge `taxi_zone_lookup.csv`, publié par la TLC, et on renomme ses colonnes en français : `zone_id`, `arrondissement`, `nom_zone` et `type_zone`. On garde uniquement les zones 1 à 263, qui sont de vraies zones de New York : les codes 264 (inconnue) et 265 (hors NYC) ne correspondent à aucun lieu réel.

Cette table sert de **base** à l'intégration : c'est elle qui garantit que les 263 zones seront toutes présentes dans le résultat, même celles où un service n'a aucune course.

**Effet sur les données :** une table de 263 lignes et 4 colonnes. On affiche aussi la répartition des zones par arrondissement.

In [2]:
zones = pd.read_csv(ZONES_CSV)
zones = zones.rename(columns={"LocationID": "zone_id", "Borough": "arrondissement",
                              "Zone": "nom_zone", "service_zone": "type_zone"})
zones = zones[zones["zone_id"].between(1, 263)].reset_index(drop=True)

print("Zones de référence :", len(zones))
display(zones.head())
display(zones["arrondissement"].value_counts().to_frame("nb_zones"))

Zones de référence : 263


,zone_id,arrondissement,nom_zone,type_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


,nb_zones
arrondissement,
Queens,69
Manhattan,69
Brooklyn,61
Bronx,43
Staten Island,20
EWR,1


## jointure des 4 services sur les 263 zones

On part de la table des 263 zones et on y joint successivement les résumés Yellow, Green, FHV et HVFHV sur la clé `zone_id`, avec une **jointure à gauche** (`how="left"`). La jointure à gauche garde toutes les zones de la table de référence, qu'elles aient des courses ou non.

Quand un service n'a aucune course dans une zone, la jointure laisse des cases vides. On les remplace par **0**, car l'absence de ligne veut dire « 0 course » et non « valeur inconnue ». Après le remplissage, on remet les colonnes de comptage (`_n`) en nombres entiers.

**Effet sur les données :** une seule table de 263 lignes et 91 colonnes : les 4 colonnes d'identification et les 87 totaux des 4 services (23 + 23 + 12 + 29).

In [3]:
integre = zones.copy()
for df in [yellow_zone, green_zone, fhv_zone, hvfhv_zone]:
    integre = integre.merge(df, on="zone_id", how="left")

colonnes_totaux = [c for c in integre.columns if c not in ["zone_id", "arrondissement", "nom_zone", "type_zone"]]
print("Valeurs vides avant remplissage :", integre[colonnes_totaux].isna().sum().sum())
integre[colonnes_totaux] = integre[colonnes_totaux].fillna(0)
colonnes_n = [c for c in colonnes_totaux if "_n" in c and "_s_" not in c]
integre[colonnes_n] = integre[colonnes_n].astype("int64")

print("Lignes :", integre.shape[0], "| colonnes :", integre.shape[1])
display(integre.head())

Valeurs vides avant remplissage : 773
Lignes : 263 | colonnes : 91


,zone_id,arrondissement,nom_zone,type_zone,yellow_n,yellow_n_2024,yellow_n_2025,yellow_n_weekend,yellow_n_nuit,yellow_n_matin,...,hvfhv_s_prix_pb,hvfhv_n_cbd,hvfhv_n_attente,hvfhv_s_attente,hvfhv_n_partage_ok,hvfhv_n_partage,hvfhv_n_pmr_ok,hvfhv_n_pmr,hvfhv_s_paie,hvfhv_s_prix_paie
0,1,EWR,Newark Airport,EWR,1338,659,679,486,90,163,...,0.00,0,0,0.00,0,0,0,0,0.00,0.00
1,2,Queens,Jamaica Bay,Boro Zone,19,11,8,6,2,4,...,155.98,0,5,31.95,5,0,5,0,165.87,155.98
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone,862,245,617,276,135,275,...,170618.30,101,8961,41715.35,9058,374,9058,50,141022.74,170564.40
3,4,Manhattan,Alphabet City,Yellow Zone,35875,12311,23564,20846,14135,4535,...,309362.20,5930,12535,63896.02,12691,813,12691,40,233212.86,309329.86
4,5,Staten Island,Arden Heights,Boro Zone,3,1,2,1,1,0,...,29491.69,12,1145,7382.20,1179,85,1179,2,24068.36,29491.69


## contrôles de l'intégration

On vérifie que la jointure n'a ni perdu ni dupliqué de courses : pour chaque service, le nombre de courses dans la table intégrée doit être **identique** à celui du résumé d'origine.

On crée ensuite la colonne `total_n`, qui additionne les courses des 4 services pour chaque zone. On liste les zones qui n'ont **aucune course**, tous services confondus (en général des îles, des parcs ou des zones industrielles sans habitants), et on compte, pour chaque service, combien de zones il ne dessert pas du tout.

**Effet sur les données :** une colonne de plus, `total_n`. Les contrôles ne modifient rien d'autre.

In [4]:
for nom, df in [("yellow", yellow_zone), ("green", green_zone), ("fhv", fhv_zone), ("hvfhv", hvfhv_zone)]:
    avant = df[nom + "_n"].sum()
    apres = integre[nom + "_n"].sum()
    print(nom, ": avant", avant, "| après", apres, "| identique :", avant == apres)

integre["total_n"] = integre["yellow_n"] + integre["green_n"] + integre["fhv_n"] + integre["hvfhv_n"]
print("\nCourses totales, 4 services :", int(integre["total_n"].sum()))

sans_course = integre[integre["total_n"] == 0]
print("Zones sans aucune course :", len(sans_course))
display(sans_course[["zone_id", "nom_zone", "arrondissement"]])

for nom in ["yellow", "green", "fhv", "hvfhv"]:
    print("Zones sans course", nom, ":", int((integre[nom + "_n"] == 0).sum()))

yellow : avant 14269897 | après 14269897 | identique : True
green : avant 208490 | après 208490 | identique : True
fhv : avant 1044984 | après 1044984 | identique : True
hvfhv : avant 4048744 | après 4048744 | identique : True

Courses totales, 4 services : 19572115
Zones sans aucune course : 2


,zone_id,nom_zone,arrondissement
102,103,Governor's Island/Ellis Island/Liberty Island,Manhattan
103,104,Governor's Island/Ellis Island/Liberty Island,Manhattan


Zones sans course yellow : 4
Zones sans course green : 23
Zones sans course fhv : 3
Zones sans course hvfhv : 4


## sauvegarde de la table intégrée

On sauvegarde la table dans `agg_data/zones_integre.csv`. Elle contient encore uniquement des **totaux** : aucune moyenne ni aucun ratio n'a été calculé. Le calcul des features (parts, moyennes, taux, croissance) se fera dans l'étape suivante, à partir de cette table.

**Effet sur les données :** aucun changement. Le fichier final fait 263 lignes et 92 colonnes (4 d'identification, 87 totaux et `total_n`).

In [5]:
cible = os.path.join(AGG_DIR, "zones_integre.csv")
integre.to_csv(cible, index=False)
print("Sauvegardé :", cible)
print("Lignes :", integre.shape[0], "| colonnes :", integre.shape[1])

Sauvegardé : c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\agg_data\zones_integre.csv
Lignes : 263 | colonnes : 92
